# AI-sfa 이펙트 LoRA 확인 그림만 다시 뽑기 (Kaggle T4)
1차 학습(kaggle_lora_train)은 성공했지만 확인 그림 단계가 LyCORIS 오류(`flag not set`)로 비었다. 학습 결과 zip(Drive 공유 링크)에서 LoRA 3개를 받아, 기본 모델 / 2·4·6바퀴를 같은 글·같은 시드로 그린다.
- 설정: **GPU T4 x2**, **Internet On** → **Save Version → Save & Run All (Commit)**. 20~40분 예상.
- 끝나면 Output의 `samples_result.zip`(그림만, 수십 MB)을 받아 Drive 공유 링크를 알려 준다.

In [ ]:
# 1) 학습 환경: 서버에서 불러오기까지 확인한 조합을 별도 환경(/tmp/env)에 그대로 깐다(Kaggle 기본 패키지와 섞이지 않게)
!pip -q install uv
!uv venv -q /tmp/env --python 3.10
!uv pip install -q --python /tmp/env/bin/python torch==2.4.1 torchvision==0.19.1 --index-url https://download.pytorch.org/whl/cu124
!git clone -q --depth 1 --branch v0.9.1 https://github.com/kohya-ss/sd-scripts.git /tmp/sd-scripts
!cd /tmp/sd-scripts && uv pip install -q --python /tmp/env/bin/python -r requirements.txt
!uv pip install -q --python /tmp/env/bin/python lycoris-lora==3.1.1 "numpy==1.26.4"
!cd /tmp/sd-scripts && /tmp/env/bin/python -c "import torch, library, lycoris, bitsandbytes, cv2; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())"

In [ ]:
# 2) 기본 모델(Hugging Face)과 1차 학습 결과(LoRA 3개, Drive 공유 링크)
!curl -sSL -C - -o /tmp/ilxl.safetensors https://huggingface.co/OnomaAIResearch/Illustrious-XL-v1.0/resolve/main/Illustrious-XL-v1.0.safetensors
!curl -sSL -o /tmp/r.zip "https://drive.usercontent.google.com/download?id=1mWFaG9QN8Bk0U1CDZRfg_0NSDDeoI290&export=download&confirm=t"
!mkdir -p /kaggle/working && cd /kaggle/working && unzip -q -o /tmp/r.zip 'out/*' && rm /tmp/r.zip && ls -la /kaggle/working/out /tmp/ilxl.safetensors

In [ ]:
# 5) 확인 그림: 같은 글·같은 시드로 기본 모델만 / 바퀴별 LoRA를 그려 비교(Output samples/)
import glob, os
prompts = [
 "nkmr_fx, anime screencap, explosion, smoke, billowing mass, white core orange rim, layered cel shading, no outline, sharp tips, expanding, large effect, no humans, simple background, wide shot",
 "nkmr_fx, anime screencap, lightning, embers, streaks, particles, white core blue rim, no outline, sharp tips, rapid expansion, no humans, city, wide shot",
 "nkmr_fx, anime screencap, impact frame, flash, radial, monochrome, rough brush strokes, peak, full screen effect, no humans, abstract background",
 "nkmr_fx, anime screencap, debris, dust cloud, billowing mass, flat color, simple cel shading, dispersing, flying, no humans, rocky ground, wide shot",
 "nkmr_fx, anime screencap, energy beam, shockwave, ring, white core pink rim, layered cel shading, sustained, large effect, no humans, sky",
 "nkmr_fx, anime screencap, fire, smoke, pillar, white core orange rim, layered cel shading, rough brush strokes, expanding, anime character, city, from below",
]
open('/tmp/prompts.txt','w').write("\n".join(p + " --n lowres, worst quality, text, watermark --w 1344 --h 768 --s 28 --l 6 --d 1234" for p in prompts))
ckpts = sorted(glob.glob('/kaggle/working/out/nkmr_fx_ilxl*.safetensors'))
print(ckpts)
runs = [("base", None)] + [(os.path.basename(c)[:-12], c) for c in ckpts]
for name, c in runs:
    out = f"/kaggle/working/samples/{name}"
    os.makedirs(out, exist_ok=True)
    net = f"--network_module lycoris.kohya --network_weights {c} --network_mul 1.0 --network_merge" if c else ""  # LyCORIS 3.1.1은 apply_to가 flag not set으로 멈춘다(2026-10-25 Kaggle 1차) → 합쳐서 그림
    os.system(f"cd /tmp/sd-scripts && /tmp/env/bin/python sdxl_gen_img.py --ckpt /tmp/ilxl.safetensors --outdir {out} --from_file /tmp/prompts.txt --images_per_prompt 1 --fp16 --no_half_vae --sdpa --sampler euler_a {net} > {out}/log.txt 2>&1")
    print(name, len(glob.glob(out + '/*.png')), 'images')

In [ ]:
# 4) 한 장으로(줄 = 기본/바퀴, 칸 = 글) + 묶기. LoRA 파일은 이미 Drive에 있으니 그림만 묶는다
from PIL import Image, ImageDraw
order = ["base", "nkmr_fx_ilxl-000002", "nkmr_fx_ilxl-000004", "nkmr_fx_ilxl"]
names = {"base": "base", "nkmr_fx_ilxl-000002": "ep2", "nkmr_fx_ilxl-000004": "ep4", "nkmr_fx_ilxl": "ep6"}
grid = [(names[r], sorted(glob.glob(f'/kaggle/working/samples/{r}/*.png'))[:6]) for r in order]
grid = [(n, ims) for n, ims in grid if ims]
W = Image.new('RGB', (448 * 6, 256 * len(grid)), 'black')
d = ImageDraw.Draw(W)
for y, (n, ims) in enumerate(grid):
    for x, p in enumerate(ims):
        W.paste(Image.open(p).resize((448, 256)), (448 * x, 256 * y))
    d.rectangle((0, 256 * y, 60, 256 * y + 22), fill='black'); d.text((4, 256 * y + 4), n, fill='white')
W.save('/kaggle/working/samples_grid.jpg', quality=88)
print([(n, len(ims)) for n, ims in grid])
!cd /kaggle/working && rm -rf out && zip -q -r samples_result.zip samples samples_grid.jpg && ls -la /kaggle/working